In [11]:
print("ram ram")

ram ram


In [12]:
# Install dependencies if you haven't already:
# !pip install langgraph langchain langchain-core langchain-litellm sqlalchemy pandas pyodbc matplotlib typing_extensions

import os
import uuid
import litellm
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from typing import Annotated, Literal, Sequence, Optional
from typing_extensions import TypedDict
from IPython.display import display, clear_output

from langchain_litellm import ChatLiteLLM
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage

# Suppress LiteLLM verbose logs
litellm.set_verbose = False

In [13]:
# Ensure API keys are available in your environment
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY")

# ── Primary LLM: Gemini (via LiteLLM Gateway) ──
orchestrator_llm_primary = ChatLiteLLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    temperature=0
)

# ── Fallback LLM: Groq LLaMA 3 (via LiteLLM Gateway) ──
orchestrator_llm_fallback = ChatLiteLLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0
)

_ACTIVE_PROVIDER = {"orchestrator": "gemini"}

def invoke_with_fallback(primary_llm, fallback_llm, messages: list, role: str, tools=None):
    """LLM Gateway router: Tries primary, falls back to secondary on error."""
    global _ACTIVE_PROVIDER
    llm_primary  = primary_llm.bind_tools(tools)  if tools else primary_llm
    llm_fallback = fallback_llm.bind_tools(tools) if tools else fallback_llm

    try:
        result = llm_primary.invoke(messages)
        if _ACTIVE_PROVIDER[role] != "gemini":
            print(f"\n[GATEWAY] {role.title()} recovered to primary (Gemini).")
        _ACTIVE_PROVIDER[role] = "gemini"
        return result
        
    except Exception as e:
        err_str = str(e)
        print(f"\n[GATEWAY WARNING] Primary failed for {role}: {err_str[:120]}")
        print(f"[GATEWAY] Routing to Fallback (Groq) for {role}...")
        _ACTIVE_PROVIDER[role] = "groq"
        
        try:
            result = llm_fallback.invoke(messages)
            print(f"[GATEWAY SUCCESS] Fallback responded successfully for {role}.")
            return result
        except Exception as e2:
            raise RuntimeError(
                f"GATEWAY CRITICAL: Both Primary and Fallback failed.\nError: {str(e2)[:200]}"
            ) from e2

print("✅ LiteLLM Gateway initialized — Primary: Gemini | Fallback: Groq")

✅ LiteLLM Gateway initialized — Primary: Gemini | Fallback: Groq


In [14]:
# Database Configuration for local SQL Server
DB_DRIVER     = os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server")
DB_SERVER     = os.getenv("DB_SERVER", "localhost")
DB_NAME       = os.getenv("DB_NAME", "mes_new")
DB_TRUSTED    = os.getenv("DB_TRUSTED_CONNECTION", "yes")
DB_ENCRYPT    = os.getenv("DB_ENCRYPT", "yes")
DB_TRUST_CERT = os.getenv("DB_TRUST_SERVER_CERTIFICATE", "yes")

conn_str = (
    f"DRIVER={{{DB_DRIVER}}};SERVER={DB_SERVER};DATABASE={DB_NAME};"
    f"Trusted_Connection={DB_TRUSTED};Encrypt={DB_ENCRYPT};"
    f"TrustServerCertificate={DB_TRUST_CERT};"
)
connection_url = f"mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}"

try:
    engine = create_engine(connection_url, echo=False)
    print(f"✅ Connected to SQL Server — Database: {DB_NAME}")
except Exception as e:
    engine = None
    print(f"⚠️ Database connection failed. Running in mocked mode. Error: {e}")

✅ Connected to SQL Server — Database: mes_new


In [15]:
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional
from sqlalchemy import text
from langchain_core.tools import tool

# ==========================================
# HELPER FUNCTION
# ==========================================


@tool
def execute_read_only_sql(sql_query: str) -> str:
    """
    Executes a custom read-only T-SQL SELECT query on the mes_new database. 
    Use this tool for general database queries when no other specific data-fetching tool is available.
    Always limit results using TOP 50.
    """
    # Security check to prevent accidental modifications
    blocked_keywords = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "TRUNCATE", "EXEC", "EXECUTE", "CREATE", "MERGE"]
    if any(keyword in sql_query.upper() for keyword in blocked_keywords):
        return "Error: Security block. Only read-only SELECT queries are allowed."
    
    return _execute_query(sql_query)

def _execute_query(query: str, params: dict = None) -> str:
    """Executes a SQL SELECT query safely with parameters and returns JSON."""
    if engine is None:
        return "Error: Database engine not connected."
    try:
        with engine.connect() as conn:
            df = pd.read_sql_query(text(query), conn, params=params or {})
        if df.empty:
            return "No matching records found."
        return df.head(50).to_json(orient="records", date_format="iso", indent=2)
    except Exception as e:
        return f"Database Error: {str(e)}"

# ==========================================
# 1. VISUALIZATION TOOLS (Charts & Graphs)
# ==========================================

@tool
def chart_top_machines_by_capacity(labels: list[str], capacities: list[float], utilizations: list[float]) -> str:
    """Generates a grouped bar chart showing the capacity and utilization of the top 5 machines."""
    fig, ax = plt.subplots(figsize=(8, 5))
    x = range(len(labels))
    ax.bar([i - 0.2 for i in x], capacities, 0.4, label='Capacity', color='#1f77b4')
    ax.bar([i + 0.2 for i in x], utilizations, 0.4, label='Utilization %', color='#ff7f0e')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.legend()
    plt.title("Top Machines: Capacity vs Utilization")
    plt.tight_layout()
    plt.show()
    return "Capacity vs Utilization chart displayed successfully."

@tool
def chart_opex_and_margins(months: list[str], opex: list[float], margins: list[float]) -> str:
    """Plots OPEX and Profit Margins over time."""
    plt.figure(figsize=(8, 5))
    plt.plot(months, opex, marker='o', label='OPEX', color='red')
    plt.plot(months, margins, marker='s', label='Profit Margin', color='green')
    plt.title("Financial Metrics: OPEX vs Profit Margins")
    plt.legend()
    plt.tight_layout()
    plt.show()
    return "OPEX and margins line chart generated."

@tool
def chart_inventory_levels(materials: list[str], quantities: list[float]) -> str:
    """Generates a bar chart of top inventory levels for raw materials."""
    plt.figure(figsize=(7, 4))
    plt.bar(materials, quantities, color='#2ca02c')
    plt.title("Top Raw Material Inventory Levels")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    return "Inventory levels chart generated."

@tool
def chart_sales_forecast_vs_actual(periods: list[str], forecast: list[float], actual: list[float]) -> str:
    """Plots a bar chart comparing sales forecasts with actual sales order quantities."""
    fig, ax = plt.subplots(figsize=(8, 5))
    x = range(len(periods))
    ax.bar([i - 0.2 for i in x], forecast, 0.4, label='Forecast', color='gray')
    ax.bar([i + 0.2 for i in x], actual, 0.4, label='Actual', color='blue')
    ax.set_xticks(x)
    ax.set_xticklabels(periods)
    ax.legend()
    plt.title("Sales: Forecast vs Actual")
    plt.tight_layout()
    plt.show()
    return "Sales forecast vs actual chart generated."

@tool
def chart_planned_vs_actual_duration(work_orders: list[str], planned_hrs: list[float], actual_hrs: list[float]) -> str:
    """Generates a chart showing planned vs actual duration for completed work orders."""
    fig, ax = plt.subplots(figsize=(8, 5))
    x = range(len(work_orders))
    ax.bar([i - 0.2 for i in x], planned_hrs, 0.4, label='Planned (Hrs)', color='lightblue')
    ax.bar([i + 0.2 for i in x], actual_hrs, 0.4, label='Actual (Hrs)', color='darkblue')
    ax.set_xticks(x)
    ax.set_xticklabels(work_orders, rotation=45)
    ax.legend()
    plt.title("Work Orders: Planned vs Actual Duration")
    plt.tight_layout()
    plt.show()
    return "Planned vs Actual duration chart generated."

@tool
def chart_material_dispatch_history(dates: list[str], quantities: list[float]) -> str:
    """Plots a line or bar chart of total dispatched material quantities over the last week."""
    plt.figure(figsize=(8, 4))
    plt.bar(dates, quantities, color='purple')
    plt.title("Material Dispatches Over Last 7 Days")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    return "Material dispatch chart generated."

@tool
def chart_machine_efficiency(machines: list[str], efficiency: list[float]) -> str:
    """Generates a chart of efficiency percentages for packaging/CNC machines."""
    plt.figure(figsize=(7, 4))
    plt.barh(machines, efficiency, color='teal')
    plt.title("Machine Efficiency Percentages")
    plt.xlim(0, 100)
    plt.tight_layout()
    plt.show()
    return "Machine efficiency chart generated."

@tool
def chart_rejection_reasons(reasons: list[str], counts: list[int]) -> str:
    """Plots the top reasons for material rejection."""
    plt.figure(figsize=(7, 7))
    plt.pie(counts, labels=reasons, autopct='%1.1f%%', startangle=140)
    plt.title("Top Reasons for Material Rejection")
    plt.tight_layout()
    plt.show()
    return "Rejection reasons pie chart generated."

@tool
def chart_operator_utilization(work_centers: list[str], utilization: list[float]) -> str:
    """Generates a chart of operator utilization across different work centers."""
    plt.figure(figsize=(8, 4))
    plt.bar(work_centers, utilization, color='orange')
    plt.title("Operator Utilization by Work Center")
    plt.ylabel("Utilization %")
    plt.tight_layout()
    plt.show()
    return "Operator utilization chart generated."

@tool
def chart_scrap_generation(dates: list[str], scrap_qty: list[float]) -> str:
    """Plots the amount of scrap generated over time."""
    plt.figure(figsize=(8, 4))
    plt.plot(dates, scrap_qty, marker='x', color='brown', linestyle='--')
    plt.title("Scrap Generation Trend")
    plt.ylabel("Quantity")
    plt.tight_layout()
    plt.show()
    return "Scrap generation chart generated."

# ==========================================
# 2. WORK ORDER TOOLS
# ==========================================

@tool
def fetch_active_work_orders() -> str:
    """Gets a breakdown of active work orders and their progress from WorkOrder table."""
    query = "SELECT WorkOrderNumber, PlannedQty, CompletedQty, ProgressPercent, Status FROM dbo.WorkOrder WHERE Status NOT IN ('Completed', 'Cancelled')"
    return _execute_query(query)

@tool
def fetch_draft_work_orders() -> str:
    """Counts or lists work orders currently in 'Draft' status."""
    query = "SELECT WorkOrderNumber, PlannedQty, DueDate FROM dbo.WorkOrder WHERE Status = 'Draft'"
    return _execute_query(query)

@tool
def fetch_in_progress_work_orders() -> str:
    """Lists all work orders that are currently 'In Progress'."""
    query = "SELECT WorkOrderNumber, MachineId, ProgressPercent FROM dbo.WorkOrder WHERE Status = 'In Progress'"
    return _execute_query(query)

@tool
def fetch_work_order_history(product_id: int) -> str:
    """Fetches the work order history for a specific Product ID."""
    query = "SELECT WorkOrderNumber, Status, CompletedQty, ActualStart, ActualEnd FROM dbo.WorkOrder WHERE ProductId = :pid ORDER BY ActualEnd DESC"
    return _execute_query(query, {"pid": product_id})

@tool
def fetch_delayed_high_priority_work_orders() -> str:
    """Checks if any high-priority work orders are delayed based on DueDate."""
    query = "SELECT WorkOrderNumber, PriorityId, DueDate, Status FROM dbo.WorkOrder WHERE DueDate < GETUTCDATE() AND Status NOT IN ('Completed', 'Cancelled')"
    return _execute_query(query)

@tool
def fetch_gantt_schedule_progress() -> str:
    """Shows the current Gantt schedule progress for active orders from GanttSchedule table."""
    query = "SELECT WorkOrderId, StartDate, EndDate, Progress, BarStatus FROM dbo.GanttSchedule"
    return _execute_query(query)

@tool
def fetch_routing_steps(routing_template_id: int) -> str:
    """Gets active routing steps for a production plan from RoutingStep table."""
    query = "SELECT Seq, Operation, WorkCenter, Machine, TotalTime FROM dbo.RoutingStep WHERE RoutingTemplateId = :rt_id"
    return _execute_query(query, {"rt_id": routing_template_id})

@tool
def fetch_routing_templates() -> str:
    """Shows active routing templates."""
    query = "SELECT TemplateCode, TemplateName, Status FROM dbo.RoutingTemplate WHERE Status = 'Active'"
    return _execute_query(query)

@tool
def fetch_work_order_bom_lines(work_order_id: int) -> str:
    """Summarizes BOM lines for a specific Work Order."""
    query = "SELECT ComponentId, Quantity, IssuedQty, ConsumedQty, Status FROM dbo.WorkOrderBOM WHERE WorkOrderId = :wo_id"
    return _execute_query(query, {"wo_id": work_order_id})

# ==========================================
# 3. MACHINES & MAINTENANCE TOOLS
# ==========================================

@tool
def fetch_top_machines_by_capacity() -> str:
    """Gets the top 5 machines by capacity from MachineMaster."""
    query = "SELECT TOP 5 MachineCode, MachineName, Capacity, Utilization FROM dbo.MachineMaster ORDER BY Capacity DESC"
    return _execute_query(query)

@tool
def fetch_machine_breakdowns_and_maintenance() -> str:
    """Provides a report on machine breakdowns and maintenance activities."""
    query = "SELECT MachineId, MaintenanceType, StartDate, EndDate, Status FROM dbo.MaintenanceWindow ORDER BY StartDate DESC"
    return _execute_query(query)

@tool
def fetch_upcoming_maintenance_windows() -> str:
    """Gets the status of upcoming scheduled maintenance windows."""
    query = "SELECT MachineId, MaintenanceType, StartDate, Status FROM dbo.MaintenanceWindow WHERE StartDate > GETUTCDATE() AND Status = 'Scheduled'"
    return _execute_query(query)

@tool
def fetch_maintenance_status() -> str:
    """Checks if any machine is currently in 'Maintenance' status."""
    query = "SELECT MachineCode, MachineName FROM dbo.MachineMaster WHERE Status = 'Maintenance'"
    return _execute_query(query)

@tool
def fetch_overdue_maintenance_windows() -> str:
    """Identifies active maintenance windows exceeding their planned duration."""
    query = "SELECT MachineId, MaintenanceType, StartDate, EndDate FROM dbo.MaintenanceWindow WHERE Status = 'In Progress' AND EndDate < GETUTCDATE()"
    return _execute_query(query)

@tool
def fetch_tooling_master_life() -> str:
    """Shows tooling master life used versus total life."""
    query = "SELECT ToolCode, ToolName, LifeUsed, LifeTotal, Status FROM dbo.ToolingMaster"
    return _execute_query(query)

@tool
def fetch_weighing_machine_outputs() -> str:
    """Summarizes outputs from the WeighingMachine table."""
    query = "SELECT MachineCode, FGQuantity, RejectedQuantity, ScrapQuantity FROM dbo.WeighingMachine WHERE IsActive = 1"
    return _execute_query(query)

@tool
def fetch_capacity_overloads() -> str:
    """Checks for active capacity analysis overload flags."""
    query = "SELECT MachineId, Date, UtilizationPercent FROM dbo.CapacityAnalysis WHERE OverloadFlag = 1"
    return _execute_query(query)

@tool
def fetch_shift_calendar() -> str:
    """Shows the shift calendar and available hours for today."""
    query = "SELECT Date, ShiftId, IsWorkingDay, AvailableHours FROM dbo.ShiftCalendar WHERE Date = CAST(GETUTCDATE() AS DATE)"
    return _execute_query(query)

@tool
def fetch_operator_shortages() -> str:
    """Identifies operator shortages in the current shift based on OperatorMaster and ShiftMaster."""
    query = "SELECT ShiftCode, OperatorCount, Status FROM dbo.ShiftMaster WHERE Status = 'Active'"
    return _execute_query(query)

# ==========================================
# 4. INVENTORY & MATERIALS TOOLS
# ==========================================

@tool
def fetch_finished_goods_stock() -> str:
    """Fetches the current inventory stock for finished goods."""
    query = "SELECT MaterialCode, MaterialName, FGQuantity, BlockedQuantity, Status FROM dbo.FinishedGood WHERE IsActive = 1"
    return _execute_query(query)

@tool
def fetch_raw_materials_below_minimum() -> str:
    """Identifies raw materials falling below their minimum stock levels."""
    query = """
        SELECT i.MaterialCode, m.MaterialName, i.QuantityInStock, m.MinimumStock 
        FROM dbo.Inventory i JOIN dbo.Materials m ON i.MaterialCode = m.MaterialCode 
        WHERE i.QuantityInStock < m.MinimumStock
    """
    return _execute_query(query)

@tool
def fetch_material_stock_by_code(material_code: str) -> str:
    """Gets the current stock level for a specific material code (e.g., RM-001)."""
    query = "SELECT MaterialCode, QuantityInStock, BlockedQuantity FROM dbo.Inventory WHERE MaterialCode = :mcode"
    return _execute_query(query, {"mcode": material_code})

@tool
def fetch_scrap_generated_last_7_days() -> str:
    """Calculates how much scrap was generated in the last 7 days from the Scrap table."""
    query = "SELECT MaterialCode, SUM(ScrapQuantity) as TotalScrap FROM dbo.Scrap WHERE OutputDate >= DATEADD(day, -7, GETUTCDATE()) GROUP BY MaterialCode"
    return _execute_query(query)

@tool
def fetch_rejected_materials_this_month() -> str:
    """Summarizes rejected materials for the current month from the Rejected table."""
    query = "SELECT MaterialCode, MaterialName, SUM(RejectedQuantity) as TotalRejected FROM dbo.Rejected WHERE MONTH(OutputDate) = MONTH(GETUTCDATE()) GROUP BY MaterialCode, MaterialName"
    return _execute_query(query)

@tool
def fetch_jumbo_packaging_inventory() -> str:
    """Gets the current status of jumbo packaging inventory."""
    query = "SELECT PackagingType, TotalQuantity, Status FROM dbo.JumboPackaging"
    return _execute_query(query)

@tool
def fetch_sfg_wip_storage() -> str:
    """Summarizes semi-finished goods (SFG) quantities currently in WIP storage."""
    query = "SELECT SFGCode, SFGName, SUM(SFGQuantity) as TotalSFG FROM dbo.WIPStorage GROUP BY SFGCode, SFGName"
    return _execute_query(query)

@tool
def fetch_online_holds() -> str:
    """Gets a summary of online holds and the affected materials."""
    query = "SELECT MaterialCode, MaterialName, OnlineHoldQuantity, Status FROM dbo.OnlineHold"
    return _execute_query(query)

@tool
def fetch_total_blocked_quantity() -> str:
    """Calculates the total blocked quantity in inventory."""
    query = "SELECT SUM(BlockedQuantity) as TotalBlocked FROM dbo.Inventory"
    return _execute_query(query)

@tool
def fetch_unpack_stage_bags() -> str:
    """Finds how many bags are currently waiting in the unpack stage."""
    query = "SELECT MaterialCode, NumberOfBags, Status FROM dbo.UnPack WHERE Status = 'Pending'"
    return _execute_query(query)

@tool
def fetch_expired_inventory_lots() -> str:
    """Checks InventoryByLot for any expired products based on LotMaster."""
    query = "SELECT l.LotNumber, l.ExpiryDate, i.Quantity FROM dbo.LotMaster l JOIN dbo.InventoryByLot i ON l.LotId = i.LotId WHERE l.ExpiryDate < CAST(GETUTCDATE() AS DATE)"
    return _execute_query(query)

@tool
def fetch_raw_material_inwards_current_week() -> str:
    """Visualizes/fetches raw material inwards for the current week."""
    query = "SELECT ItemCode, ItemName, ReceivedQuantity, RMInwardDate FROM dbo.RMInward WHERE RMInwardDate >= DATEADD(wk, DATEDIFF(wk, 7, GETUTCDATE()), 0)"
    return _execute_query(query)

# ==========================================
# 5. SALES, DISPATCH & VENDOR TOOLS
# ==========================================

@tool
def fetch_sales_order_details(so_number: str) -> str:
    """Fetches full details for a specific Sales Order (e.g., SO-1001)."""
    query = "SELECT SONumber, CustomerCode, MaterialCode, OrderQuantity, Status, TotalAmount FROM dbo.SalesOrder WHERE SONumber = :so"
    return _execute_query(query, {"so": so_number})

@tool
def fetch_material_dispatch_history() -> str:
    """Gets the material dispatch history for the current month."""
    query = "SELECT SONumber, Quantity, DispatchDate, Status FROM dbo.MaterialDispatch WHERE MONTH(DispatchDate) = MONTH(GETUTCDATE())"
    return _execute_query(query)

@tool
def fetch_delayed_material_dispatches() -> str:
    """Checks if there are any delayed material dispatches."""
    query = "SELECT SONumber, OrderQuantity, PlannedDispatchDate FROM dbo.SalesOrder WHERE PlannedDispatchDate < GETUTCDATE() AND Status NOT IN ('Dispatched', 'Delivered')"
    return _execute_query(query)

@tool
def fetch_top_salespersons() -> str:
    """Shows the total sales achieved by the top salespersons."""
    query = "SELECT FullName, TotalSales, TargetSales, AchievedSales FROM dbo.SalesPerson ORDER BY CAST(AchievedSales AS FLOAT) DESC"
    return _execute_query(query)

@tool
def fetch_purchase_returns() -> str:
    """Gets the latest purchase returns and their reasons."""
    query = "SELECT PONumber, ItemCode, ReturnQty, PurchaseReturnDate FROM dbo.PurchaseReturn ORDER BY PurchaseReturnDate DESC"
    return _execute_query(query)

@tool
def fetch_dispatched_return_data() -> str:
    """Fetches return data for dispatched materials this quarter."""
    query = "SELECT DispatchNumber, QuantityReturned, Remarks, Date FROM dbo.ReturnData"
    return _execute_query(query)

@tool
def fetch_vehicle_loading_details() -> str:
    """Shows vehicle loading details for today's dispatches."""
    query = "SELECT VehicleNumber, LoadQuantity, LoadedBags, Status FROM dbo.VehicleDetails WHERE CAST(CreatedOn AS DATE) = CAST(GETUTCDATE() AS DATE)"
    return _execute_query(query)

# ==========================================
# 6. PLANNING & PROCUREMENT TOOLS
# ==========================================

@tool
def fetch_mrp_run_results() -> str:
    """Summarizes MRP run results and suggested quantities."""
    query = "SELECT ProductId, ActionType, SuggestedQty, SuggestedDate FROM dbo.MRP_Result WHERE IsAccepted = 0"
    return _execute_query(query)

@tool
def fetch_pending_mps_requirements() -> str:
    """Gets pending raw material requirements for this week's MPS."""
    query = "SELECT RMCode, RMName, RequiredQty, ShortageQty FROM dbo.MPSRawMaterialRequirement WHERE Status = 'Pending'"
    return _execute_query(query)

@tool
def fetch_weekly_planning_status() -> str:
    """Checks the status of the weekly planning requirements."""
    query = "SELECT MaterialCode, WeekNumber, PlannedQty, RequiredProduction FROM dbo.WeeklyPlanning"
    return _execute_query(query)

@tool
def fetch_tentative_rm_plan() -> str:
    """Summarizes the tentative raw material plan for the next month."""
    query = "SELECT Segment, RMGrade, FinalRequirement, MonthYear FROM dbo.TentativeRMPlan"
    return _execute_query(query)

@tool
def fetch_open_purchase_orders() -> str:
    """Gets the total amount/details of purchase orders currently open."""
    query = "SELECT PONumber, VendorCode, Quantity, Status FROM dbo.PODetails WHERE Status = 'Open'"
    return _execute_query(query)

@tool
def fetch_vendor_details() -> str:
    """Pulls vendor details (e.g., for main plastic suppliers)."""
    query = "SELECT VendorName, VendorCode, PhoneNumber, EmailAddress FROM dbo.Vendors WHERE IsActive = 1"
    return _execute_query(query)

@tool
def fetch_rm_invoices_sent_to_erp() -> str:
    """Checks the status of RM invoices sent to the ERP."""
    query = "SELECT PONumber, InvoiceNumber, SendToERP, SendOn FROM dbo.RMInvoice"
    return _execute_query(query)

@tool
def fetch_pending_wip_rm_requests() -> str:
    """Plots/Fetches WIP RM requests that are still pending."""
    query = "SELECT RequestId, ItemCode, RequestedQuantity, Status FROM dbo.WIPRMRequest WHERE Status = 'Pending'"
    return _execute_query(query)

# ==========================================
# 7. ALERTS, QA & SYSTEM LOGS TOOLS
# ==========================================

@tool
def fetch_active_alerts_by_severity() -> str:
    """Lists all active alerts sorted by severity from AlertMaster."""
    query = "SELECT AlertType, Severity, Title, CreatedDate FROM dbo.AlertMaster WHERE IsResolved = 0 ORDER BY Severity DESC"
    return _execute_query(query)

@tool
def fetch_qa_alerts_today() -> str:
    """Summarizes quality assurance alerts triggered today."""
    query = "SELECT AlertType, Message, CreatedDate FROM dbo.AlertMaster WHERE Source = 'QA' AND CAST(CreatedDate AS DATE) = CAST(GETUTCDATE() AS DATE)"
    return _execute_query(query)

@tool
def fetch_unresolved_system_logs() -> str:
    """Checks for unresolved system event logs for work orders."""
    query = "SELECT WorkOrderId, Title, Description FROM dbo.WorkOrderLog WHERE IsSystemGenerated = 1 AND IsResolved = 0"
    return _execute_query(query)

# ==========================================
# 8. GENERAL & ADMIN TOOLS
# ==========================================

@tool
def fetch_pending_workstep_actions() -> str:
    """Provides a summary of current workstep actions pending approval."""
    query = "SELECT RequestId, ActionName, TargetStepId FROM dbo.WorkStepActions"
    return _execute_query(query)

@tool
def fetch_workflow_history() -> str:
    """Summarizes the workflow history for recently closed requests."""
    query = "SELECT RequestId, WorkStepName, ActionName, ActionDate FROM dbo.WorkflowHistory ORDER BY ActionDate DESC"
    return _execute_query(query)

@tool
def fetch_hsn_codes() -> str:
    """Fetches the latest HSN codes added to the system."""
    query = "SELECT HSNCodeName, CreatedOn FROM dbo.HSNCode ORDER BY CreatedOn DESC"
    return _execute_query(query)

@tool
def fetch_excise_classifications() -> str:
    """Summarizes the excise classifications of current stock."""
    query = "SELECT ExciseClassificationName FROM dbo.ExciseClassification"
    return _execute_query(query)

@tool
def fetch_customer_product_mappings() -> str:
    """Summarizes customer product mappings that were recently updated."""
    query = "SELECT CustomerId, ProductId, UpdatedDate FROM dbo.CustomerProductMapping_del WHERE IsActive = 1 ORDER BY UpdatedDate DESC"
    return _execute_query(query)

@tool
def handle_general_conversation(message: str) -> str:
    """
    Use this tool ONLY when the user is making general conversation, saying hello, 
    or asking off-topic questions that do not require database data.
    """
    return "No database data needed. Reply directly and politely to the user, sir."


# Combine all tools
# Combine all 50+ tools for the Executive Insights Agent
mes_tools = [
    # 1. Visualization Tools
    chart_top_machines_by_capacity, chart_opex_and_margins, chart_inventory_levels, 
    chart_sales_forecast_vs_actual, chart_planned_vs_actual_duration, 
    chart_material_dispatch_history, chart_machine_efficiency, chart_rejection_reasons, 
    chart_operator_utilization, chart_scrap_generation,

    # 2. Work Order Tools
    fetch_active_work_orders, fetch_draft_work_orders, fetch_in_progress_work_orders, 
    fetch_work_order_history, fetch_delayed_high_priority_work_orders, 
    fetch_gantt_schedule_progress, fetch_routing_steps, fetch_routing_templates, 
    fetch_work_order_bom_lines,

    # 3. Machines & Maintenance Tools
    fetch_top_machines_by_capacity, fetch_machine_breakdowns_and_maintenance, 
    fetch_upcoming_maintenance_windows, fetch_maintenance_status, 
    fetch_overdue_maintenance_windows, fetch_tooling_master_life, 
    fetch_weighing_machine_outputs, fetch_capacity_overloads, fetch_shift_calendar, 
    fetch_operator_shortages,

    # 4. Inventory & Materials Tools
    fetch_finished_goods_stock, fetch_raw_materials_below_minimum, 
    fetch_material_stock_by_code, fetch_scrap_generated_last_7_days, 
    fetch_rejected_materials_this_month, fetch_jumbo_packaging_inventory, 
    fetch_sfg_wip_storage, fetch_online_holds, fetch_total_blocked_quantity, 
    fetch_unpack_stage_bags, fetch_expired_inventory_lots, 
    fetch_raw_material_inwards_current_week,

    # 5. Sales, Dispatch & Vendor Tools
    fetch_sales_order_details, fetch_material_dispatch_history, 
    fetch_delayed_material_dispatches, fetch_top_salespersons, fetch_purchase_returns, 
    fetch_dispatched_return_data, fetch_vehicle_loading_details,

    # 6. Planning & Procurement Tools
    fetch_mrp_run_results, fetch_pending_mps_requirements, fetch_weekly_planning_status, 
    fetch_tentative_rm_plan, fetch_open_purchase_orders, fetch_vendor_details, 
    fetch_rm_invoices_sent_to_erp, fetch_pending_wip_rm_requests,

    # 7. Alerts, QA & System Logs Tools
    fetch_active_alerts_by_severity, fetch_qa_alerts_today, fetch_unresolved_system_logs,

    # 8. General & Admin Tools
    fetch_pending_workstep_actions, fetch_workflow_history, fetch_hsn_codes, 
    fetch_excise_classifications, fetch_customer_product_mappings,
    
    execute_read_only_sql,
    handle_general_conversation
]

# Bind the expanded toolset to the ToolNode
tool_node = ToolNode(mes_tools)

In [16]:
# 1. Define State
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# 2. System Prompt
SYSTEM_PROMPT = """You are Deva, an Executive Insights & Smart IIIoT MES Assistant created by IIIOT InfoTech. 
Address the user as 'sir'.

INSTRUCTIONS:
1. If the user is just making general conversation, greeting you, or asking casual questions, simply reply to them politely and naturally without using any tools.
2. When the user asks for manufacturing data (inventory, machines, work orders), use the available tools to fetch the data.
3. CRITICAL: If the user asks to "visualize", "plot", or "chart" data, you MUST use the `generate_bar_chart` tool to display it, and then provide a short narrative summary.
"""

# 3. Agent Node
def agent_node(state: AgentState):
    messages = state["messages"]
    
    # Inject system prompt
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
        
    response = invoke_with_fallback(
        primary_llm=orchestrator_llm_primary,
        fallback_llm=orchestrator_llm_fallback,
        messages=messages,
        role="orchestrator",
        tools=mes_tools
    )
    return {"messages": [response]}

# 4. Router Logic
def route_tools(state: AgentState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "__end__"

# 5. Build Graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", route_tools)
workflow.add_edge("tools", "agent")

# 6. Compile with Checkpointer
memory = MemorySaver()
deva_mes_agent = workflow.compile(checkpointer=memory)

print("✅ Agent Architecture compiled successfully!")

✅ Agent Architecture compiled successfully!


In [17]:
def run_chat():
    """
    Runs the chat in a continuous while loop. 
    Generates a new thread_id every time it is called.
    """
    # Renewing the thread_id on every fresh execution
    current_thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": current_thread_id}}
    
    print("=" * 60)
    print(" DEVA: EXECUTIVE INSIGHTS INITIALIZED")
    print(f"Session Thread ID: {current_thread_id}")
    print("Type 'exit', 'quit', or 'break' to close the conversation.")
    print("=" * 60 + "\n")
    
    while True:
        user_input = input("You (Sir): ")
        
        if not user_input.strip():
            continue
            
        if user_input.lower().strip() in ['exit', 'quit', 'break']:
            print("\nExiting chat. Goodbye, sir!")
            break
            
        print("\nDeva is thinking...\n")
        
        events = deva_mes_agent.stream(
            {"messages": [HumanMessage(content=user_input)]}, 
            config=config, 
            stream_mode="values"
        )
        
        # Iterate through events and print the final AI summary
        for event in events:
            if "messages" in event:
                last_msg = event["messages"][-1]
                
                # Print only the final narrative response (ignoring tool call wrappers)
                if isinstance(last_msg, AIMessage) and not last_msg.tool_calls and last_msg.content:
                    # Clear the "thinking" output for a cleaner UI (optional)
                    clear_output(wait=True)
                    print(f"Deva:\n{last_msg.content}\n")
                    print("-" * 60)

# Start the chat loop
run_chat()

Deva:
Here is the chart and breakdown of the current work orders in the system, sir:

### Work Order Breakdown:
1. **WO202608030001**: Status is **Planned** with a planned quantity of `111,111` and `0` completed.
2. **WO202608100001**: Status is **Cancelled** with a planned quantity of `2`.
3. **WO202608100002**: Status is **Draft** with a planned quantity of `8,888,888` and `0` completed.

------------------------------------------------------------

Exiting chat. Goodbye, sir!
